In [ ]:
!pip install neptune-client

In [ ]:
!pip install torchsampler

In [ ]:
!pip install Keras-Applications

In [ ]:
!pip install keras.applications
!pip install keras-resnet

In [ ]:
!pip install split-folders
!pip install pytest-shutil

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn import CrossEntropyLoss, Dropout, Softmax, Linear, Conv2d, LayerNorm
import torchvision
from torchvision import models
from torch.utils.data import DataLoader, Dataset
import torch.utils.data as utils
from torchvision import transforms

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import os
import math
import copy
import sys
from tqdm import tqdm


import neptune.new as neptune
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.utils import class_weight, shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, fbeta_score
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    classification_report,
    cohen_kappa_score,
    accuracy_score,
)
import splitfolders


from PIL import Image, ImageFilter, ImageOps


import warnings

warnings.filterwarnings("ignore")


import skimage.io
from skimage.transform import resize
from imgaug import augmenters as iaa


import tensorflow as tf
import keras
from keras.losses import binary_crossentropy
from keras_applications.resnet import ResNet50, preprocess_input
import keras.backend as K
import seaborn as sns
from torchsampler import ImbalancedDatasetSampler
#from keras.utils import Sequence
#from keras.utils import to_categorical


WORKERS = 2
CHANNEL = 3


IMG_SIZE = 512
NUM_CLASSES = 5
SEED = 77

In [ ]:
img_dir = "../input/aptos2019-blindness-detection/train_images/"

In [ ]:
data = pd.read_csv("../input/aptos2019-blindness-detection/train.csv")
data.head()

In [ ]:
# Distribution of Training Data
display(data['diagnosis'].value_counts())
sns.countplot(data['diagnosis'], color='blue')

In [ ]:
img_size = 300
def crop_image_from_gray(img,tol=7):
    if img.ndim ==2:
        mask = img>tol
        return img[np.ix_(mask.any(1),mask.any(0))]
    elif img.ndim==3:
        gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        mask = gray_img>tol
        
        check_shape = img[:,:,0][np.ix_(mask.any(1),mask.any(0))].shape[0]
        if (check_shape == 0): # image is too dark so that we crop out everything,
            return img # return original image
        else:
            img1=img[:,:,0][np.ix_(mask.any(1),mask.any(0))]
            img2=img[:,:,1][np.ix_(mask.any(1),mask.any(0))]
            img3=img[:,:,2][np.ix_(mask.any(1),mask.any(0))]
            img = np.stack([img1,img2,img3],axis=-1)
        return img
def preprocess_image(img_file):
    img = cv2.imread(img_file)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = crop_image_from_gray(img)
    img = cv2.resize(img, (img_size,img_size))
    return img

In [ ]:
fig=plt.figure(figsize=(8, 8))
image_path = data.loc[8,'id_code']
image_id = data.loc[8,'diagnosis']
img =preprocess_image(f'../input/aptos2019-blindness-detection/train_images/{image_path}.png')
plt.title(f'diagnosis:{image_id} index:{26}')
plt.imshow(img)
plt.tight_layout()

In [ ]:
def isClear(img, threshold = 60):
    return cv2.Laplacian(img, cv2.CV_64F).var() > threshold

In [ ]:
def display_samples(df, columns=4, rows=3):
    fig=plt.figure(figsize=(5*columns, 4*rows))
    for i in range(columns*rows):
        idx = np.random.randint(0, len(df)-1, 1)[0]
        image_path = df.loc[idx,'id_code']
        image_id = df.loc[idx,'diagnosis']
        img = preprocess_image(f'../input/aptos2019-blindness-detection/train_images/{image_path}.png')
        fig.add_subplot(rows, columns, i+1)
        plt.title(f'diagnosis:{image_id}   isclear:{isClear(img)}')
        plt.imshow(img)
    plt.tight_layout()
display_samples(data)

In [ ]:
import time
blur_list = []
blur_list_id = []
start_time = time.time();
for i, image_id in enumerate(tqdm(data['id_code'])):
    img = preprocess_image(f'../input/aptos2019-blindness-detection/train_images/{image_id}.png')
    if(not isClear(img)):
        blur_list.append(i)
        blur_list_id.append(image_id)
data = data.drop(blur_list)
print(f'Cost: {time.time() - start_time}:.3% seconds');

In [ ]:
print(f'Droped items:{len(blur_list_id)}')

In [ ]:
def display_blurry_samples(df, img_id_list, columns=4, rows=3):
    fig=plt.figure(figsize=(5*columns, 4*rows))
    for i in range(columns*rows):
        img = preprocess_image(f'../input/aptos2019-blindness-detection/train_images/{img_id_list[i]}.png')
        fig.add_subplot(rows, columns, i+1)
        plt.title(f'index:{i}  isclear:{isClear(img)}')
        plt.imshow(img)
    plt.tight_layout()
display_blurry_samples(data, blur_list_id)

In [ ]:
data

In [ ]:
no = data[data.diagnosis == 0]
mild = data[data.diagnosis == 1]
moderate = data[data.diagnosis == 2]
severe = data[data.diagnosis == 3]
dr = data[data.diagnosis == 4]

In [ ]:
def seperation(data):
  file_path = []
  for f in range(len(data)):
      file = data.id_code.iloc[f]
      file = img_dir + file + '.png'
      file_path.append(file)
  return file_path

In [ ]:
no = seperation(no)
mild = seperation(mild)
moderate = seperation(moderate)
severe = seperation(severe)
dr = seperation(dr)

In [ ]:
len(no) + len(mild) + len(moderate) + len(severe) + len(dr) == len(data)

In [ ]:
try:
    os.mkdir('train')
    os.mkdir("train/no")
    os.mkdir("train/mild")
    os.mkdir("train/moderate")
    os.mkdir("train/severe")
    os.mkdir("train/dr")
except:
    pass

In [ ]:
import shutil
def move(path, dst='train/no'):
  for old_path in path:
      name = old_path.split('/')
      name = name[-1]
      shutil.copy(old_path, f'{dst}/{name}')
move(no)
move(mild, dst='train/mild')
move(moderate, dst='train/moderate')
move(severe, dst='train/severe')
move(dr, dst='train/dr')

In [ ]:
# bs = 1
# crop_size = 255

# transformations = torchvision.transforms.Compose([
#         torchvision.transforms.ToTensor(),
#         torchvision.transforms.Resize(crop_size),
#         torchvision.transforms.CenterCrop(crop_size),
#         torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
#     ])

# train_data = torchvision.datasets.ImageFolder(root='./train', transform = transformations)
# train_data = torch.utils.data.DataLoader(train_data, shuffle=True, batch_size=bs)

In [ ]:
# train_data.dataset.classes

In [ ]:
import splitfolders
# data_dir = '/content/train/'

splitfolders.ratio('./train/', output="output", seed=1337, ratio=(.8, 0.1,0.1))

In [ ]:
train_dir = './output/train'
len(train_dir)

In [ ]:
!rm -r -f ./train

In [ ]:
def crop_image1(img,tol=7):
    # img is image data
    # tol  is tolerance
        
    mask = img>tol
    return img[np.ix_(mask.any(1),mask.any(0))]

def crop_image_from_gray(img,tol=7):
    if img.ndim ==2:
        mask = img>tol
        return img[np.ix_(mask.any(1),mask.any(0))]
    elif img.ndim==3:
        gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        mask = gray_img>tol
        
        check_shape = img[:,:,0][np.ix_(mask.any(1),mask.any(0))].shape[0]
        if (check_shape == 0): # image is too dark so that we crop out everything,
            return img # return original image
        else:
            img1=img[:,:,0][np.ix_(mask.any(1),mask.any(0))]
            img2=img[:,:,1][np.ix_(mask.any(1),mask.any(0))]
            img3=img[:,:,2][np.ix_(mask.any(1),mask.any(0))]
    #         print(img1.shape,img2.shape,img3.shape)
            img = np.stack([img1,img2,img3],axis=-1)
    #         print(img.shape)
        return img

In [ ]:
def load_ben_color1(image, sigmaX=50):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = crop_image_from_gray(image)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
    image=cv2.addWeighted (image,4, cv2.GaussianBlur( image , (0,0) , sigmaX) ,-4 ,128)
        
    return image

def circle_crop1(img, sigmaX=10):   
    """
    Create circular crop around image centre    
    """    
    
    
    image=load_ben_color1(img,sigmaX=50)
    
#     height, width, depth = img.shape    
    
#     x = int(width/2)
#     y = int(height/2)
#     r = np.amin((x,y))
    
#     circle_img = np.zeros((height, width), np.uint8)
#     cv2.circle(circle_img, (x,y), int(r), 1, thickness=-1)
#     img = cv2.bitwise_and(img, img, mask=circle_img)
    #img = crop_image_from_gray(img)
#     img=cv2.addWeighted ( img,4, cv2.GaussianBlur( img , (0,0) , sigmaX) ,-4 ,128)
    return img

In [ ]:
!mkdir preprocessed
!mkdir ./preprocessed/train
!mkdir ./preprocessed/train/no
!mkdir ./preprocessed/train/mild
!mkdir ./preprocessed/train/moderate
!mkdir ./preprocessed/train/dr
!mkdir ./preprocessed/train/severe

In [ ]:
import os
from os import listdir
import shutil

list1=['dr', 'mild', 'moderate', 'no', 'severe']
for i in list1:
    folder_dir = "./output/train/"
    path1=os.path.join(folder_dir,i)
    for images in os.listdir(path1):
 
    # check if the image ends with png
        if (images.endswith(".png")):
            imagePath=os.path.join(path1,images)
            filename=images
#             print(imagePath)
            #img = cv2.imread(imagePath)
    
    
            image = cv2.imread(imagePath)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
#         image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
            image=cv2.addWeighted ( image,4, cv2.GaussianBlur( image , (0,0) , IMG_SIZE/10) ,-4 ,128) # the trick is to add this line

    
    
    
    
    
            #image=circle_crop1(img)
            
            dest_dir= "./preprocessed/train/"
            path2=os.path.join(dest_dir,i,filename)
            
            #print(path2)
            cv2.imwrite(path2,image)
#             break
            #print(image)
            
            #result=PIL.Image.fromarray(image)
            #print(result)
            #result = PIL.Image.fromarray((image * 255).astype(numpy.uint8))
            #resultsave=result.save(filename)
            #print(resultsave)
            #picture = picture.save(filename) 
            #image_file=cv2.imwrite(image,filename)
            #shutil.copy(resultsave, path2)

In [ ]:
%pylab inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
img = mpimg.imread(r'./preprocessed/train/mild/db4ed1e07aa3.png')
imgplot = plt.imshow(img)
plt.show()

In [ ]:
# !mkdir preprocessed
!mkdir ./preprocessed/val
!mkdir ./preprocessed/val/no
!mkdir ./preprocessed/val/mild
!mkdir ./preprocessed/val/moderate
!mkdir ./preprocessed/val/dr
!mkdir ./preprocessed/val/severe

In [ ]:
import os
from os import listdir
import shutil

list1=['dr', 'mild', 'moderate', 'no', 'severe']
for i in list1:
    folder_dir = "./output/val/"
    path1=os.path.join(folder_dir,i)
    for images in os.listdir(path1):
 
    # check if the image ends with png
        if (images.endswith(".png")):
            imagePath=os.path.join(path1,images)
            filename=images
#             print(imagePath)
            #img = cv2.imread(imagePath)
    
    
            image = cv2.imread(imagePath)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
#         image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
            image=cv2.addWeighted ( image,4, cv2.GaussianBlur( image , (0,0) , IMG_SIZE/10) ,-4 ,128) # the trick is to add this line

    
    
    
    
    
            #image=circle_crop1(img)
            
            dest_dir= "./preprocessed/val/"
            path2=os.path.join(dest_dir,i,filename)
            
            #print(path2)
            cv2.imwrite(path2,image)
#             break
            #print(image)
            
            #result=PIL.Image.fromarray(image)
            #print(result)
            #result = PIL.Image.fromarray((image * 255).astype(numpy.uint8))
            #resultsave=result.save(filename)
            #print(resultsave)
            #picture = picture.save(filename) 
            #image_file=cv2.imwrite(image,filename)
            #shutil.copy(resultsave, path2)

In [ ]:
# !mkdir preprocessed
!mkdir ./preprocessed/test
!mkdir ./preprocessed/test/no
!mkdir ./preprocessed/test/mild
!mkdir ./preprocessed/test/moderate
!mkdir ./preprocessed/test/dr
!mkdir ./preprocessed/test/severe

In [ ]:
import os
from os import listdir
import shutil

list1=['dr', 'mild', 'moderate', 'no', 'severe']
for i in list1:
    folder_dir = "./output/test/"
    path1=os.path.join(folder_dir,i)
    for images in os.listdir(path1):
 
    # check if the image ends with png
        if (images.endswith(".png")):
            imagePath=os.path.join(path1,images)
            filename=images
#             print(imagePath)
            #img = cv2.imread(imagePath)
    
    
            image = cv2.imread(imagePath)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
#         image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
            image=cv2.addWeighted ( image,4, cv2.GaussianBlur( image , (0,0) , IMG_SIZE/10) ,-4 ,128) # the trick is to add this line

    
    
    
    
    
            #image=circle_crop1(img)
            
            dest_dir= "./preprocessed/test/"
            path2=os.path.join(dest_dir,i,filename)
            
            #print(path2)
            cv2.imwrite(path2,image)
#             break
            #print(image)
            
            #result=PIL.Image.fromarray(image)
            #print(result)
            #result = PIL.Image.fromarray((image * 255).astype(numpy.uint8))
            #resultsave=result.save(filename)
            #print(resultsave)
            #picture = picture.save(filename) 
            #image_file=cv2.imwrite(image,filename)
            #shutil.copy(resultsave, path2)

In [ ]:
train_dir = './preprocessed/train'
val_dir = './preprocessed/val'
test_dir = './preprocessed/test'

In [ ]:
rm -r -f ./output

In [ ]:


"""# Create Dataset"""

dtype = torch.FloatTensor
dtype = torch.cuda.FloatTensor





# train_csv = pd.read_csv("../input/aptos2019-blindness-detection/train.csv")

# # train_df, val_df = train_test_split(
# #     train_csv, test_size=0.1, stratify=train_csv.diagnosis
# )
# train_df.reset_index(drop=True, inplace=True)
# val_df.reset_index(drop=True, inplace=True)
# train_df.head()

bs = 2
crop_size = 255


def dataset(bs, crop_size, sample_size="full"):
    transformations = torchvision.transforms.Compose(
        [
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Resize(crop_size),
            torchvision.transforms.CenterCrop(crop_size),
            torchvision.transforms.Normalize(
                mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
            ),
        ]
    )

    if sample_size == "full":
        train_data = torchvision.datasets.ImageFolder(
            root=train_dir, transform=transformations
        )
        valid_data = torchvision.datasets.ImageFolder(
            root=val_dir, transform=transformations
        )
        test_data = torchvision.datasets.ImageFolder(
            root=test_dir, transform=transformations
        )

        train_data = torch.utils.data.DataLoader(
            train_data, shuffle=False, batch_size=bs,num_workers=2,sampler=ImbalancedDatasetSampler(train_data)
        )
        valid_data = torch.utils.data.DataLoader(
            valid_data, shuffle=False, batch_size=bs,num_workers=2,sampler=ImbalancedDatasetSampler(valid_data)
        )
        test_loader = torch.utils.data.DataLoader(
            test_data, shuffle=False, batch_size=bs,num_workers=2,sampler=ImbalancedDatasetSampler(test_data)
        )

    else:
        train_data = torchvision.datasets.ImageFolder(
            root=train_dir, transform=transformations
        )
        indices = torch.arange(sample_size)
        train_data = torch.utils.data.Subset(train_data, indices)
        train_data = torch.utils.data.DataLoader(
            train_data, shuffle=True, batch_size=bs,num_workers=2
        )

    valid_data = torchvision.datasets.ImageFolder(
        root=val_dir, transform=transformations
    )
    valid_data = torch.utils.data.DataLoader(valid_data, shuffle=False, batch_size=bs,num_workers=2,sampler=ImbalancedDatasetSampler(valid_data))

    test_data = torchvision.datasets.ImageFolder(
        root=test_dir, transform=transformations
    )
    test_data = torch.utils.data.DataLoader(test_data, shuffle=False, batch_size=bs,num_workers=2, sampler=ImbalancedDatasetSampler(test_data))

    return train_data, valid_data, test_data

In [ ]:
batch_size = 5

In [ ]:
classes = ('0','1','2','3','4')
train_data, valid_data, test_data = dataset(5, 255)

In [ ]:
def imshow(img):
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


# get some random training images
dataiter = iter(train_data)
images, labels = next(dataiter)

# show images
imshow(torchvision.utils.make_grid(images))
# print labels
print(' '.join(f'{classes[labels[j]]:5s}' for j in range(batch_size)))

In [ ]:
import copy

In [ ]:


"""# Data Modeling"""

from torch.nn.modules.utils import _pair


class Embeddings(nn.Module):
    """Construct the embeddings from patch, position embeddings."""

    def __init__(self, img_size: int, hidden_size: int, in_channels: int):

        super(Embeddings, self).__init__()
        img_size = _pair(img_size)

        patch_size = _pair(img_size)
        n_patches = (img_size[0] // patch_size[0]) * (img_size[1] // patch_size[1])

        self.patch_embeddings = Conv2d(
            in_channels=in_channels,
            out_channels=hidden_size,
            kernel_size=patch_size,
            stride=patch_size,
        )

        self.position_embeddings = nn.Parameter(torch.zeros(1, 2, hidden_size))
        self.cls_token = nn.Parameter(torch.zeros(1, 1, hidden_size))

        self.dropout = Dropout(0.1)

    def forward(self, x):
        B = x.shape[0]
        cls_tokens = self.cls_token.expand(B, -1, -1)

        x = self.patch_embeddings(x)
        x = x.flatten(2)
        x = x.transpose(-1, -2)
        x = torch.cat((cls_tokens, x), dim=1)

        embeddings = x + self.position_embeddings
        embeddings = self.dropout(embeddings)
        return embeddings



class Attention(nn.Module):
    def __init__(self, num_attention_heads, hidden_size, attention_dropout_rate):
        super(Attention, self).__init__()
        self.num_attention_heads = num_attention_heads
        self.attention_head_size = int(hidden_size / self.num_attention_heads)
        self.all_head_size = self.num_attention_heads * self.attention_head_size

        self.query = Linear(hidden_size, self.all_head_size)
        self.key = Linear(hidden_size, self.all_head_size)
        self.value = Linear(hidden_size, self.all_head_size)

        self.out = Linear(hidden_size, self.all_head_size)
        self.attn_dropout = Dropout(attention_dropout_rate)
        self.proj_dropout = Dropout(attention_dropout_rate)

        self.softmax = torch.nn.Softmax(dim=-1)

    def transpose_for_scores(self, x):
        new_x_shape = x.size()[:-1] + (
            self.num_attention_heads,
            self.attention_head_size,
        )
        x = x.view(*new_x_shape)
        return x.permute(0, 2, 1, 3)

    def forward(self, hidden_states):
        mixed_query_layer = self.query(hidden_states)
        mixed_key_layer = self.key(hidden_states)
        mixed_value_layer = self.value(hidden_states)

        query_layer = self.transpose_for_scores(mixed_query_layer)
        key_layer = self.transpose_for_scores(mixed_key_layer)
        value_layer = self.transpose_for_scores(mixed_value_layer)

        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)
        attention_probs = self.softmax(attention_scores)
        weights = attention_probs
        attention_probs = self.attn_dropout(attention_probs)

        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(*new_context_layer_shape)
        attention_output = self.out(context_layer)
        attention_output = self.proj_dropout(attention_output)
        return attention_output, weights


class Mlp(nn.Module):
    def __init__(self, hidden_size, linear_dim, dropout_rate, std_norm):
        super(Mlp, self).__init__()
        self.fc1 = Linear(hidden_size, linear_dim)
        self.fc2 = Linear(linear_dim, hidden_size)
        self.act_fn = torch.nn.functional.gelu
        self.dropout = Dropout(dropout_rate)
        self.std_norm = std_norm
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.normal_(self.fc1.bias, std=self.std_norm)
        nn.init.normal_(self.fc2.bias, std=self.std_norm)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act_fn(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x


class Block(nn.Module):
    def __init__(
        self,
        num_attention_heads,
        hidden_size,
        linear_dim,
        dropout_rate,
        attention_dropout_rate,
        eps,
        std_norm,
    ):

        super(Block, self).__init__()
        self.hidden_size = hidden_size
        self.attention_norm = LayerNorm(hidden_size, eps=eps)
        self.ffn_norm = LayerNorm(hidden_size, eps=eps)
        self.ffn = Mlp(
            hidden_size=hidden_size,
            linear_dim=linear_dim,
            dropout_rate=dropout_rate,
            std_norm=std_norm,
        )
        self.attn = Attention(
            num_attention_heads=num_attention_heads,
            hidden_size=hidden_size,
            attention_dropout_rate=attention_dropout_rate,
        )

    def forward(self, x):
        h = x
        x = self.attention_norm(x)
        x, weights = self.attn(x)
        x = x + h
        h = x
        x = self.ffn_norm(x)
        x = self.ffn(x)
        x = x + h
        return x, weights


class Encoder(nn.Module):
    def __init__(
        self,
        num_layers,
        hidden_size,
        num_attention_heads,
        linear_dim,
        dropout_rate,
        attention_dropout_rate,
        eps,
        std_norm,
    ):
        super(Encoder, self).__init__()
        self.layer = nn.ModuleList()
        self.encoder_norm = LayerNorm(hidden_size, eps=eps)
        for _ in range(num_layers):
            layer = Block(
                num_attention_heads,
                hidden_size,
                linear_dim,
                dropout_rate,
                attention_dropout_rate,
                eps,
                std_norm,
            )
            self.layer.append(copy.deepcopy(layer))

    def forward(self, hidden_states):
        attn_weights = []
        for layer_block in self.layer:
            hidden_states, weights = layer_block(hidden_states)
            attn_weights.append(weights)
        encoded = self.encoder_norm(hidden_states)
        return encoded, attn_weights

In [ ]:




class Transformer(nn.Module):
    def __init__(
        self,
        img_size,
        hidden_size,
        in_channels,
        num_layers,
        num_attention_heads,
        linear_dim,
        dropout_rate,
        attention_dropout_rate,
        eps,
        std_norm,
    ):
        super(Transformer, self).__init__()
        self.embeddings = Embeddings(img_size, hidden_size, in_channels)

        self.encoder = Encoder(
            num_layers,
            hidden_size,
            num_attention_heads,
            linear_dim=config.LINEAR_DIM,
            dropout_rate=config.DROPOUT_RATE,
            attention_dropout_rate=config.ATTENTION_DROPOUT_RATE,
            eps=config.EPS,
            std_norm=config.STD_NORM,
        )

    def forward(self, input_ids):
        embedding_output = self.embeddings(input_ids)
        encoded, attn_weights = self.encoder(embedding_output)
        return encoded, attn_weights


class VisionTransformer(nn.Module):
    def __init__(
        self,
        img_size,
        num_classes,
        hidden_size,
        in_channels,
        num_layers,
        num_attention_heads,
        linear_dim,
        dropout_rate,
        attention_dropout_rate,
        eps,
        std_norm,
    ):
        super(VisionTransformer, self).__init__()
        self.classifier = "token"

        self.transformer = Transformer(
            img_size,
            hidden_size,
            in_channels,
            num_layers,
            num_attention_heads,
            linear_dim,
            dropout_rate,
            attention_dropout_rate,
            eps,
            std_norm,
        )
        self.head = Linear(hidden_size, num_classes)

    def forward(self, x, labels=None):
        x, attn_weights = self.transformer(x)
        logits = self.head(x[:, 0])

        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, 400), labels.view(-1))
            return loss
        else:
            return logits, attn_weights

In [ ]:



def train_Engine(
    n_epochs,
    train_data,
    val_data,
    model,
    optimizer,
    device,
    loss_fn="nll_loss",
    monitoring=True,
):

    logits_m = []
    yo = []
    attention = []
    device = 'cuda'
    confusion_matrix = torch.zeros(5, 5)

    best_accuracy = 0
    for epoch in range(1, n_epochs + 1):
        print("Epoch:", epoch)
        for i, (data, target) in tqdm(
            enumerate(train_data), total=len(train_data), desc="Training"
        ):
            total_samples = len(train_data.dataset)
            best_accuracy = 0
            # device
            model = model.to(device)
            x = data.to(device)
            y = target.to(device)
            optimizer.zero_grad()

            logits, attn_weights = model(x)
            proba = F.log_softmax(logits, dim=1)
            loss = F.nll_loss(proba, y, reduction="sum")
            loss.backward()

            yhat = torch.argmax(logits, dim=1)
            accuracy = torch.sum(yhat == y).item() / len(train_data)
            optimizer.step()

            if monitoring:
                run["Training_loss"].log(loss.item())
                run["Training_acc"].log(accuracy)

            if accuracy > best_accuracy:
                best_model = model
                torch.save(best_model, f"./ViT.pth")

        model.eval()
        total_samples = len(val_data.dataset)
        correct_samples = 0
        total_loss = 0

        with torch.no_grad():
            for i, (data, target) in tqdm(
                enumerate(val_data), total=len(val_data), desc="Evaluation"
            ):

                model = model.to(device)
                x = data.to(device)
                y = target.to(device)

                logits, attn_weights = model(x)
                proba = F.log_softmax(logits, dim=1)
                val_loss = F.nll_loss(proba, y, reduction="sum")
                _, pred = torch.max(logits, dim=1)

                for t, p in zip(y.view(-1), pred.view(-1)):
                    confusion_matrix[t.long(), p.long()] += 1

                total_loss += val_loss.item()
                correct_samples += pred.eq(y).sum()
                val_acc = torch.sum(pred == y).item() / len(val_data)
                avg_loss = total_loss / total_samples

                if monitoring:
                    run["Val_loss"].log(avg_loss)
                    run["Val_accuracy"].log(val_acc)

    return  logits_m, yo, attention , confusion_matrix,

In [ ]:


class Config:
    # Image configuration
    IMG_SIZE = 255
    PATCH_SIZE = 10
    CROP_SIZE = 255
    BATCH_SIZE = 5
    DATASET_SAMPLE = "full"
    N_EPOCHS = 5

    # opimizer configuration
    LR = 0.003
    OPIMIZER = "Adam"

    # Model configuration
    NUM_CLASSES = 5
    IN_CHANNELS = 3
    HIDDEN_SIZE = 768
    NUM_ATTENTION_HEADS = 12
    LINEAR_DIM = 3072
    NUM_LAYERS = 12

    ATTENTION_DROPOUT_RATE = 0.1
    DROPOUT_RATE = 0.1
    STD_NORM = 1e-6
    EPS = 1e-6

    MPL_DIM = 128
    OUTPUT = "softmax"
    LOSS_FN = "nll_loss"

    # Device configuration
    DEVICE = ["cpu", "cuda"]
    

# Config.__dict__
config = Config()


x = torch.randn(1, config.IN_CHANNELS * config.IMG_SIZE * config.IMG_SIZE)
x = x.reshape(1, config.IN_CHANNELS, config.IMG_SIZE, config.IMG_SIZE)


def neptune_monitoring(config):
    PARAMS = {}
    for key, val in config.__dict__.items():
        if key not in ["__module__", "__dict__", "__weakref__", "__doc__"]:
            PARAMS[key] = val
    return PARAMS

model = VisionTransformer(
    img_size=config.IMG_SIZE,
    num_classes=config.NUM_CLASSES,
    hidden_size=config.HIDDEN_SIZE,
    in_channels=config.IN_CHANNELS,
    num_layers=config.NUM_LAYERS,
    num_attention_heads=config.NUM_ATTENTION_HEADS,
    linear_dim=config.LINEAR_DIM,
    dropout_rate=config.DROPOUT_RATE,
    attention_dropout_rate=config.ATTENTION_DROPOUT_RATE,
    eps=config.EPS,
    std_norm=config.STD_NORM,
)

optimizer = optim.Adam(model.parameters(), lr=0.003)

run = neptune.init(
    project="mukundanks/DR-702",
    api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiI3YTM2ZmU2NS1kM2RmLTRkMTAtOTJkOC1iMDk5Yzg3ZDI1M2MifQ==",
)
logits, attn_weights = model(x)

if __name__ == "__main__":

    config = Config()
    params = neptune_monitoring(Config)
    run = neptune.init(
        project="mukundanks/DR-702",
        api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiI3YTM2ZmU2NS1kM2RmLTRkMTAtOTJkOC1iMDk5Yzg3ZDI1M2MifQ==",
    )
    run["parameters"] = params

    model = VisionTransformer(
        img_size=config.IMG_SIZE,
        num_classes=config.NUM_CLASSES,
        hidden_size=config.HIDDEN_SIZE,
        in_channels=config.IN_CHANNELS,
        num_layers=config.NUM_LAYERS,
        num_attention_heads=config.NUM_ATTENTION_HEADS,
        linear_dim=config.LINEAR_DIM,
        dropout_rate=config.DROPOUT_RATE,
        attention_dropout_rate=config.ATTENTION_DROPOUT_RATE,
        eps=config.EPS,
        std_norm=config.STD_NORM,
    )

    train_data, val_data, test_data = dataset(
        config.BATCH_SIZE, config.IMG_SIZE, config.DATASET_SAMPLE
    )

    optimizer = optim.Adam(model.parameters(), lr=0.003)
    train_Engine(
        n_epochs=config.N_EPOCHS,
        train_data=train_data,
        val_data=val_data,
        model=model,
        optimizer=optimizer,
        loss_fn="nll_loss",
        device=config.DEVICE[1],
        monitoring=True,
    )

In [ ]:
"""# Test

"""

# train_data, val_data, test_data = dataset(config.BATCH_SIZE,config.IMG_SIZE, config.DATASET_SAMPLE)

#test_data

from tqdm import tqdm

m = torch.load(
    f"./ViT.pth", map_location=torch.device("cuda")
)

total_samples = len(valid_data.dataset)
correct_samples = 0
total_loss = 0

confusion_matrix = torch.zeros(5, 5)


c = 0
total = 0
device = "cuda"
n_epochs = 20
total_step = len(train_data)
iterations = 12

logits_ = []
ground = []

with torch.no_grad():
    model.eval()
    for epoch in range(1, n_epochs):
        with tqdm(test_data, unit="batch") as tepoch:
            tepoch.set_description(f"Epoch {epoch}")
            for batch_idx, (data, target) in enumerate(tepoch):
                total_samples = len(train_data.dataset)
                best_accuracy = 0
                # device
                model = model.to(device)
                x = data.to(device)
                y = target.to(device)
                logits, attn_weights = model(x)
                proba = F.log_softmax(logits, dim=1)
                loss = F.nll_loss(proba, y, reduction="sum")
                _, pred = torch.max(logits, dim=1)
                c += torch.sum(pred == y).item()
                total += target.size(0)
                tepoch.set_postfix(loss=loss.item(), accuracy=(100 * c / total))

                logits_.append(logits)
                ground.append(y)

        for t, p in zip(y.view(-1), pred.view(-1)):
            confusion_matrix[t.long(), p.long()] += 1

import seaborn as sns

plt.figure(figsize=(12, 8))
cm = sns.heatmap(confusion_matrix)
confusion = cm.figure
run["confusion_matrix-img"].upload(confusion)

probs = torch.zeros(len(logits_))
run["probability"].log(probs)

y_ = torch.zeros(len(ground))
idx = 0
for l, o in zip(logits_, ground):
    _, l = torch.max(l, dim=1)
    probs[idx] = l
    y_[idx] = o.item()
    idx += 1

prob = probs.to(torch.long).numpy()

y_ = y_.to(torch.long).numpy()


accuracy = accuracy_score(y_, prob)
run["accuracy"].log(accuracy)

kappa_score = cohen_kappa_score(y_, prob)
run["kappa_score"].log(kappa_score)

classification = classification_report(y_, prob)
run["classification"].log(classification)

precision = precision_score(y_, prob, average="micro")
run["precision"].log(precision)

f1 = f1_score(y_, prob, average="micro")
run["f1"].log(f1)

recall = recall_score(y_, prob, average="micro")
run["recall"].log(recall)

In [ ]:
test_data.dataset.classes

In [ ]:
run.stop()